# Testing Jev (TypeSafe)

This is **Part 2 of 3**: the official Python SDK. Previous: `part1_raw_http.ipynb`. Next: `part3_battle.ipynb`.

Install the libraries once (from the repo folder): `uv pip install -r requirements.txt`, then pick the `.venv` kernel.

# Part 2: The official Python SDK

The SDK reads the key from the `TYPESAFE_API_KEY` env var automatically, builds questions with `Noul` / `Choice` / `Score` classes, and returns typed objects instead of raw dicts.

In [ ]:
import os
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)

In [ ]:
from typesafe_sdk import TypeSafeClient, Noul, Choice, Score

client = TypeSafeClient()  # uses TYPESAFE_API_KEY from the environment

## 1. List models

In [ ]:
for m in client.models.list().models:
    print(f"{m.name}: {m.description}")

## 2. Same three questions, SDK style

In [ ]:
state = "I was charged twice for my subscription this month. Please refund the extra charge."

questions = {
    "billing": Noul(instructions="Is this message about billing?"),
    "tone": Choice(
        instructions="What is the tone of this message?",
        criteria={
            "angry": "Hostile or upset",
            "neutral": "Calm and matter-of-fact",
            "happy": "Positive",
        },
    ),
    "urgency": Score(
        instructions="How urgent is this message?",
        criteria=["Not urgent", "Somewhat urgent", "Very urgent"],
    ),
}

result = client.system_one(state, questions, model="jev-latest")

## 3. Read the answers

Answers are grouped by type: `result.nouls`, `result.choices`, `result.scores`. Each one is an object, so you use attributes (`.choice`) instead of dict keys (`['choice']`).

In [ ]:
print(f"Model used: {result.model}")
print(f"Billing (yes probability): {result.nouls['billing'].noul}")
print(f"Tone: {result.choices['tone'].choice} (confidence {result.choices['tone'].confidence})")
print(f"Tone probabilities: {result.choices['tone'].probabilities}")
print(f"Urgency score: {result.scores['urgency'].score} (confidence {result.scores['urgency'].confidence})")
print(f"Tokens: {result.usage}")

## Comparing the two

| | Raw HTTP | SDK |
|---|---|---|
| Setup | just `requests` | `pip install typesafe-sdk` |
| Auth | build the `Authorization` header yourself | reads `TYPESAFE_API_KEY` for you |
| Questions | plain dicts with a `"type"` field | `Noul`, `Choice`, `Score` classes |
| Answers | `result["answers"]["tone"]["choice"]` | `result.choices["tone"].choice` |
| Extras | none | retries, async client, typed errors |